### KPI-> READMISSION (Eligible,Readmission,Readmission_rate)

In [0]:
WITH seq AS (
  SELECT 
    pateint_id,
    encounter_id,
    start_time,
    end_time,

    LAG(end_time) OVER (
      PARTITION BY pateint_id 
      ORDER BY start_time
    ) as prev_end

  FROM medical_catalog.gold.fact
)

, flags AS (
  SELECT *,

    CASE 
      WHEN prev_end IS NOT NULL 
           AND start_time >= prev_end
      THEN 1 ELSE 0 
    END as eligible,

    CASE 
        WHEN prev_end IS NOT NULL AND start_time >= prev_end 
           AND DATEDIFF(start_time, prev_end) <= 30
      THEN 1 ELSE 0 
    END as readmission

  FROM seq
)
SELECT 
  c.year,
  c.month,

  SUM(eligible) as eligible,
  SUM(readmission) as readmissions,

  ROUND(
    SUM(readmission) * 100.0 / SUM(eligible),
    2
  ) as rate

FROM flags f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date

GROUP BY c.year, c.month
ORDER BY c.year, c.month;


